# 02. Supervisor 패턴 — 핸드오프 도구 방식

> 이 노트북은 Supervisor가 Worker 에이전트를 조율할 때 **핸드오프 도구**(`transfer_to_X`)를 호출하는 방식을 다뤄요. 같은 Supervisor 아키텍처를 **구조화 출력**(`RouteResponse(next=...)`) 방식으로 구현하는 레슨은 **`04-Multi-Agent-Supervisor.ipynb`** 에 있어요 — 둘은 경쟁이 아니라 **같은 패턴의 두 구현 전략**이에요.

| 축 | 02 (이 노트북) | 04-Multi-Agent-Supervisor |
|----|----------------|---------------------------|
| 다음 노드 선택 방법 | 핸드오프 **도구 호출** | **구조화 출력** 라우팅 |
| 핵심 API | `create_supervisor`, `create_handoff_tool`, `Command(goto=...)` | `RouteResponse(next: Literal[...])`, `with_structured_output` |
| 제어 흐름 | LLM이 도구를 통해 Command 반환 | LLM이 스키마에 맞춰 직접 라우트 결정 |
| 장점 | 표준 tool-calling 모델과 자연스러움 | 라우팅 메타데이터를 깔끔하게 관찰 가능 |

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. `langgraph-supervisor` 라이브러리를 활용하여 Supervisor 에이전트를 빠르게 구현할 수 있어요
2. `create_handoff_tool`과 `Command` 패턴을 이해하고 커스텀 Supervisor를 직접 만들 수 있어요
3. `task_description` 핸드오프로 명시적 작업 지시를 에이전트에 전달할 수 있어요
4. `StateGraph`로 Supervisor + Worker 에이전트 구조를 직접 조립할 수 있어요

## 사전 지식

- 이전 노트북: `01-Multi-Agent-Overview.ipynb`에서 배운 5가지 멀티에이전트 패턴
- `create_agent`로 에이전트 생성하는 방법 (Part 5 참고)
- LangGraph `StateGraph`, `Command`, `START/END` 기초

## Supervisor 패턴이란?

**Supervisor 패턴**은 복잡한 작업을 여러 전문 에이전트에게 분배하고 조정하는 멀티에이전트 아키텍처예요.

> 💡 **핵심 정리**: Supervisor 패턴은 실제 회사의 **프로젝트 매니저(PM)**와 같아요. PM은 직접 코딩하거나 디자인하지 않아요. 대신 "이 작업은 백엔드팀에서, 저 작업은 프론트엔드팀에서" 하고 분배하고, 결과를 취합해서 보고하죠. Supervisor도 마찬가지로 직접 작업을 수행하지 않고, 핸드오프 도구로 적합한 전문가에게 **위임**만 해요.

### 핵심 구성 요소

| 구성 요소 | 역할 | 비유 | 특징 |
|-----------|------|------|------|
| **Supervisor** | 작업 위임 및 조정 | 프로젝트 매니저 | 핸드오프 도구로 워커 에이전트 호출 |
| **Worker Agent** | 전문 작업 실행 | 팀원 (개발자, 디자이너) | 자신의 도메인에만 집중 |
| **Handoff Tool** | 제어권 전달 | 업무 지시서 | `Command(goto=..., graph=PARENT)` 패턴 |
| **create_supervisor** | 빠른 구성 | 팀 빌더 키트 | `langgraph-supervisor` 라이브러리 제공 |

> 🔑 **핵심 개념**: Supervisor 패턴에서 에이전트 간 제어권 이동을 **핸드오프(Handoff)**라고 해요. "이 업무는 A팀에서 처리하고, 완료되면 B팀으로 넘기세요" 같은 방식이에요.

### 아키텍처 다이어그램

```mermaid
flowchart TD
    User([사용자 요청<br/>User Request])
    Supervisor[Supervisor<br/>작업 분석 및 위임]
    Research[Research Agent<br/>웹 검색 및 정보 수집]
    Math[Math Agent<br/>수학 계산 및 분석]
    Result([최종 응답<br/>Final Response])

    User --> Supervisor
    Supervisor -- 핸드오프 --> Research
    Supervisor -- 핸드오프 --> Math
    Research -- 결과 반환 --> Supervisor
    Math -- 결과 반환 --> Supervisor
    Supervisor --> Result

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class User,Result input
    class Supervisor process
    class Research,Math process
```

## 환경 설정

In [ ]:
# 환경 변수 로드 (.env 파일에서 API 키를 가져와요)
from dotenv import load_dotenv

load_dotenv(override=True)
# 환경 변수 로드 완료

In [ ]:
# LangSmith 추적 설정 (실행 흐름을 시각화해서 확인할 수 있어요)
import os

# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-08-Supervisor"

# LangSmith 추적 설정 완료

## 헬퍼 함수 정의

여러 에이전트가 동시에 작업하는 멀티에이전트 시스템에서는 각 에이전트의 출력을 구분해서 보기 어려워요.
노드별로 구분선을 추가하여 가독성을 높이는 헬퍼 함수를 먼저 정의할게요.

In [ ]:
from langchain_core.messages import convert_to_messages

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 메시지 출력 헬퍼 함수 정의
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 1. 워커 에이전트 생성

Supervisor에게 위임받을 전문 워커 에이전트를 먼저 만들어요.
두 가지 전문 에이전트를 만들 거예요:

1. **Research Agent**: Tavily 검색 API로 웹에서 정보를 수집해요
2. **Math Agent**: 사칙연산, 백분율 등 수학 계산을 수행해요

> 🔑 **핵심 개념**: `create_agent`의 `name` 파라미터가 매우 중요해요. 이 이름이 LangGraph 그래프의 **노드 이름**으로 사용되고, Supervisor가 핸드오프할 때 이 이름을 사용해요.

In [ ]:
import warnings
from langchain_tavily import TavilySearch
from langchain.agents import create_agent

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: Research Agent 생성 (웹 검색 전문)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: Math Agent 생성 (수학 계산 전문)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


워커 에이전트가 잘 동작하는지 간단히 테스트해볼게요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: Math Agent 단독 테스트
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 2. create_supervisor로 빠르게 구현하기

`langgraph-supervisor` 라이브러리는 Supervisor 패턴을 빠르게 구현할 수 있도록 도와줘요.
`create_supervisor` 함수 하나로 핸드오프 도구 생성, 에이전트 연결, 그래프 구성이 자동으로 처리돼요.

> 💡 **실무 팁**: `langgraph-supervisor`는 별도 패키지예요. 설치하려면 `pip install langgraph-supervisor`를 실행하세요. V1 기준으로 이 패키지가 권장 방법이에요.

> ⚠️ **자주 하는 실수**: `output_mode="last_message"`로 설정하면 최종 메시지만 반환돼요. 에이전트 실행 과정을 모두 보고 싶다면 `output_mode="full_history"`를 사용하세요.

In [ ]:
from langgraph_supervisor import create_supervisor
from langchain.chat_models import init_chat_model

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: langgraph-supervisor 라이브러리로 Supervisor 생성
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → supervisor → (research_agent | math_agent) → supervisor → ... → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


Research와 Math 에이전트를 모두 필요로 하는 복합 질문으로 테스트해볼게요.
Supervisor가 두 에이전트에게 순서대로 위임하는 과정을 확인할 수 있어요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: Supervisor 복합 작업 테스트
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 3. 커스텀 Supervisor 직접 구현하기

이번에는 라이브러리 없이 Supervisor를 직접 만들어볼게요.
내부 동작 원리를 이해하면 더 복잡한 시스템도 자유롭게 설계할 수 있어요.

### 핸드오프 작동 방식

```mermaid
sequenceDiagram
    participant S as Supervisor
    participant H as Handoff Tool
    participant R as Research Agent

    S->>H: transfer_to_research_agent 호출
    H->>H: Command(goto="research_agent", graph=PARENT) 생성
    H-->>S: Command 반환
    Note over S,R: 그래프가 research_agent 노드로 이동
    R->>R: 작업 수행
    R-->>S: supervisor 노드로 복귀 (엣지 설정)
```

> 🔑 **핵심 개념**: `InjectedState`와 `InjectedToolCallId`는 LLM에게 노출되지 않는 주입(Inject) 마커예요. LLM이 호출할 때 이 파라미터는 보이지 않고, LangGraph가 자동으로 현재 상태와 도구 호출 ID를 주입해줘요.

> ⚠️ **자주 하는 실수**: `graph=Command.PARENT`를 빠뜨리면 안 돼요. 이 설정이 없으면 서브그래프 안에서 라우팅이 발생해요. 부모 그래프 레벨에서 에이전트 간 이동이 이루어지도록 반드시 명시해야 해요.

In [ ]:
from typing import Annotated
from langchain.tools import tool
from langchain_core.tools import InjectedToolCallId
from langgraph.prebuilt import InjectedState
from langgraph.graph import MessagesState
from langgraph.types import Command

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 커스텀 핸드오프 도구 팩토리 함수 정의
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langgraph.graph import StateGraph, START, END

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 커스텀 Supervisor 에이전트와 멀티에이전트 그래프 구성
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → supervisor → (research_agent | math_agent) → supervisor → ... → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 커스텀 Supervisor 테스트 — 순수 수학 문제로 핸드오프 메커니즘 검증
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 4. 명시적 작업 지시 핸드오프 (Task Description Handoff)

지금까지는 Supervisor가 전체 메시지 히스토리를 에이전트에게 전달했어요.
하지만 메시지가 많아질수록 에이전트가 무엇을 해야 할지 찾기 어려워져요.

**Task Description Handoff**는 Supervisor가 명시적인 작업 설명을 직접 작성해서 전달하는 방식이에요.

> 💡 **핵심 정리**: 일반 핸드오프는 **팀장이 회의록 전체를 팀원에게 보내는 것**과 같아요. Task Description 핸드오프는 **핵심 요약만 적어 보내는 것**이에요. "회의 내용 중에 GDP 데이터를 찾아서 비율을 계산해줘"라고 명확하게 지시하면, 팀원이 100페이지 회의록을 읽지 않아도 바로 작업에 착수할 수 있어요.

### 기존 방식 vs Task Description 방식

| 방식 | 에이전트가 받는 내용 | 비유 | 장점 | 단점 |
|------|---------------------|------|------|------|
| **기존 방식** | 전체 대화 히스토리 | 회의록 전체 전달 | 맥락 풍부 | 불필요한 내용 포함 |
| **Task Description** | Supervisor가 쓴 명시적 지시 | 핵심만 요약한 업무 지시서 | 정확하고 간결 | 지시 작성 비용 |

> 💡 **실무 팁**: `Send` 패턴을 사용하면 에이전트에게 전달할 상태를 완전히 새롭게 구성할 수 있어요. 에이전트는 전체 대화 히스토리 대신 Supervisor가 요약한 명확한 지시만 받아요. 이를 통해 에이전트 성능이 향상돼요.

In [ ]:
from langgraph.types import Send

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 에이전트에게 전달할 새로운 상태 (히스토리를 작업 설명으로 교체)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: Task Description 방식의 고급 Supervisor 그래프 구성
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → supervisor → (research_agent | math_agent) → supervisor → ... → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 고급 Supervisor 테스트 (subgraphs=True로 내부 실행도 추적)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 5. 실습: 나만의 Supervisor 만들기

지금까지 배운 내용을 바탕으로 직접 커스텀 Supervisor를 만들어봐요.
아래 TODO 블록을 수정해서 새로운 워커 에이전트를 추가해보세요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 번역가 에이전트를 추가하여 3-에이전트 Supervisor를 만들어보세요
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 6. 실전 활용: 고객 지원 시스템

Supervisor 패턴을 실제 비즈니스 시나리오에 적용해볼게요.
3가지 전문 에이전트로 구성된 고객 지원 시스템이에요:

- **기술 지원 에이전트**: 시스템 오류, 연결 문제 처리
- **청구 지원 에이전트**: 인보이스, 결제, 환불 처리
- **일반 지원 에이전트**: 제품 정보, 영업 시간 등 안내

> 💡 **실무 팁**: Tool Calling 패턴(에이전트를 `@tool`로 래핑)도 Supervisor 패턴처럼 유용해요. 하위 에이전트를 단순히 함수처럼 호출하고 싶을 때 적합하고, 상태 관리가 단순한 경우에 사용해요.

In [ ]:
from langchain.chat_models import init_chat_model

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 고객 지원 시스템 - 에이전트 및 도구 정의
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langgraph_supervisor import create_supervisor

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 고객 지원 Supervisor 생성 (create_supervisor 라이브러리 활용)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → supervisor → (tech_support | billing_support | general_support) → supervisor → ... → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 고객 지원 시스템 테스트 - 3가지 유형의 문의
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## Supervisor 구현 방식 비교

이 노트북에서 배운 세 가지 방식을 비교해볼게요:

| 방식 | 코드 양 | 유연성 | 추천 상황 |
|------|---------|--------|----------|
| **create_supervisor** | 적음 | 중간 | 빠른 프로토타이핑, 표준 패턴 |
| **커스텀 StateGraph** | 중간 | 높음 | 커스텀 로직, 세밀한 제어 |
| **Task Description** | 중간 | 높음 | 에이전트별 명확한 지시가 필요할 때 |

> 💡 **핵심 정리**: 세 방식 모두 내부적으로는 동일한 원리(`Command` + 핸드오프)로 동작해요. `create_supervisor`는 편의 함수일 뿐이고, 커스텀 구현은 더 많은 제어권을 줘요. 실무에서는 처음에 `create_supervisor`로 시작하고, 필요할 때 커스텀 구현으로 전환하는 것을 추천해요.

## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **create_supervisor**: `langgraph-supervisor` 라이브러리의 편의 함수로, 에이전트 목록만 넘기면 Supervisor를 자동 구성해줘요
- **create_handoff_tool**: Supervisor에서 Worker로 제어권을 전달하는 핸드오프 도구 팩토리 패턴이에요
- **Command(goto=, graph=PARENT)**: 에이전트 간 이동을 제어하는 핵심 메커니즘이에요. `graph=PARENT`는 부모 그래프에서 라우팅이 이루어지도록 해요
- **InjectedState / InjectedToolCallId**: LLM에게 노출되지 않고 자동으로 주입되는 파라미터로, 핸드오프 도구의 핵심 구성 요소예요
- **Task Description Handoff**: `Send` 패턴으로 Supervisor가 직접 작성한 명시적 작업 지시를 에이전트에게 전달해요
- **StateGraph 직접 조립**: `add_node`, `add_edge`로 Supervisor와 Worker 에이전트를 연결하는 그래프를 직접 구성할 수 있어요

## 다음 노트북 예고

다음 `03-Multi-Agent-Collaboration.ipynb`에서는 **에이전트 간 도구 기반 협업**을 배워요. Supervisor 없이 에이전트들이 서로를 도구로 호출하며 협력하는 패턴을 살펴볼게요.